# 00 Data Loading and Preprocessing

## Purpose

This notebook loads the raw paper dataset and creates the final cleaned datasets used in later notebooks.

The main preprocessing steps are:

1. Load the raw JSON/JSONL paper data.
2. Extract first-author and last-author names.
3. Extract first-author, last-author, and all-author OpenAlex IDs.
4. Infer first-author and last-author gender using `nomquamgender`.
5. Create first-last author gender categories: `MM`, `MW`, `WM`, `WW`, and categories containing `U` for unknown.
6. Prepare reusable analysis columns: publication year, decade, taxonomy fields, venue rank, and number of references.
7. Construct internal citation edges from reference lists.
8. Remove self-citations.
9. Remove isolated papers after edge filtering.
10. Save final paper-level and citation-edge datasets.

## Inputs

- `data/raw/cs_paper_data.jsonl` or `data/raw/cs_paper_data.json`

## Outputs

- `data/processed/papers_with_gender.pkl`
- `data/processed/papers_final.pkl`
- `data/processed/citation_edges_final.pkl`
- `data/processed/citation_edges_all_to_cs_final.pkl`
- `data/processed/citation_edges_cs_to_cs_final.pkl`
- `data/processed/cs_papers_prepared.pkl`
- `data/processed/cs_papers_final.pkl`
- `data/processed/preprocessing_filter_summary.csv`

Notes:

- `papers_with_gender.pkl` is kept for backwards compatibility, but it now contains the final active paper set after citation-edge filtering.
- Coauthor-neighborhood citation removal is not applied here yet, because it requires a precise methodological decision about whether the neighborhood should be time-aware. A disabled scaffold is included in the filtering section.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import nomquamgender as nqg
from __future__ import annotations

In [2]:
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

Project root: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance
Raw data dir: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\raw
Processed data dir: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed


## 1. Helper Functions

In [3]:
def pick_author_name(author: dict) -> str | None:
    """
    Extract a stable author name from the author dictionary.

    Preference:
    1. dblp_id, if available
    2. name, otherwise

    Returns None if no usable string is available.
    """
    if not isinstance(author, dict):
        return None

    candidate = author.get("dblp_id") or author.get("name")

    if not isinstance(candidate, str):
        return None

    candidate = candidate.strip()
    return candidate if candidate else None

In [4]:
def first_last_author_names(author_list):
    """
    Extract first-author and last-author names from a paper's author list.
    """
    if not isinstance(author_list, list) or len(author_list) == 0:
        return None, None

    first_author = pick_author_name(author_list[0])
    last_author = pick_author_name(author_list[-1])

    return first_author, last_author

In [5]:
def pick_openalex_author_id(author: dict) -> str | None:
    """
    Extract OpenAlex author ID from the author dictionary.
    """
    if not isinstance(author, dict):
        return None

    value = author.get("openalex_id")

    if isinstance(value, str) and value.strip():
        return value.strip()

    return None

In [6]:
def first_last_author_openalex_ids(author_list):
    """
    Extract first-author and last-author OpenAlex IDs from a paper's author list.
    """
    if not isinstance(author_list, list) or len(author_list) == 0:
        return None, None

    first_author_id = pick_openalex_author_id(author_list[0])
    last_author_id = pick_openalex_author_id(author_list[-1])

    return first_author_id, last_author_id

In [7]:
def all_author_openalex_ids(author_list):
    """
    Extract all available OpenAlex author IDs from a paper's author list.

    Returns a tuple instead of a list so that the values are stable and lightweight
    when stored in the dataframe.
    """
    if not isinstance(author_list, list) or len(author_list) == 0:
        return tuple()

    ids = []
    for author in author_list:
        author_id = pick_openalex_author_id(author)
        if author_id is not None:
            ids.append(author_id)

    return tuple(dict.fromkeys(ids))


In [8]:
def p_to_label(p, lo=0.3, hi=0.7):
    """
    Convert nomquamgender p(gf) score into a gender label.

    p(gf) >= hi -> female
    p(gf) <= lo -> male
    otherwise   -> unknown

    The default thresholds are intentionally conservative.
    """
    if pd.isna(p):
        return "unknown"

    if p >= hi:
        return "female"

    if p <= lo:
        return "male"

    return "unknown"

In [9]:
def pair_cat(first_gender, last_gender):
    """
    Convert first-author and last-author gender labels into a compact category.

    female  -> W
    male    -> M
    unknown -> U

    Examples:
    male, male       -> MM
    male, female     -> MW
    female, male     -> WM
    female, female   -> WW
    unknown, male    -> UM
    """
    mapping = {
        "female": "W",
        "male": "M",
        "unknown": "U"
    }

    first = mapping.get(first_gender, "U")
    last = mapping.get(last_gender, "U")

    return first + last

## 2. Load Raw Data

In [10]:
jsonl_path = RAW_DIR / "cs_paper_data.jsonl"
json_path = RAW_DIR / "cs_paper_data.json"

if jsonl_path.exists():
    raw_path = jsonl_path
elif json_path.exists():
    raw_path = json_path
else:
    raise FileNotFoundError(
        f"No raw data file found. Expected one of:\n"
        f"- {jsonl_path}\n"
        f"- {json_path}"
    )

print("Using raw data file:", raw_path)

Using raw data file: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\raw\cs_paper_data.jsonl


In [11]:
if raw_path.suffix == ".jsonl":
    df = pd.read_json(raw_path, lines=True)
else:
    df = pd.read_json(raw_path)
print("Number of papers:", len(df))

Number of papers: 3720575


## 3. Extract First and Last Authors

In [12]:
author_lists = df["author_lst"].tolist()

first_names = []
last_names = []

for author_list in author_lists:
    first_author, last_author = first_last_author_names(author_list)
    first_names.append(first_author)
    last_names.append(last_author)

df["first_author_name"] = first_names
df["last_author_name"] = last_names

print("First author name coverage:", df["first_author_name"].notna().mean())
print("Last author name coverage:", df["last_author_name"].notna().mean())

df[["first_author_name", "last_author_name"]].head()

First author name coverage: 1.0
Last author name coverage: 1.0


,first_author_name,last_author_name
0,Christian Sallaberry,Mauro Gaio
1,Sébastien Herbreteau,Charles Kervrann
2,Sergio Venturini,Raffaella Piccarreta
3,Laura Laguna Salvadó,Bartel A. Van de Walle
4,Zahra Bouanani-Oukhaled,Laurent Breda


In [13]:
first_author_ids = []
last_author_ids = []

for author_list in author_lists:
    first_author_id, last_author_id = first_last_author_openalex_ids(author_list)
    first_author_ids.append(first_author_id)
    last_author_ids.append(last_author_id)

df["first_author_openalex_id"] = first_author_ids
df["last_author_openalex_id"] = last_author_ids

print("First author OpenAlex ID coverage:", df["first_author_openalex_id"].notna().mean())
print("Last author OpenAlex ID coverage:", df["last_author_openalex_id"].notna().mean())

df[[
    "first_author_name",
    "first_author_openalex_id",
    "last_author_name",
    "last_author_openalex_id"
]].head()

First author OpenAlex ID coverage: 1.0
Last author OpenAlex ID coverage: 1.0


,first_author_name,first_author_openalex_id,last_author_name,last_author_openalex_id
0,Christian Sallaberry,A5046716771,Mauro Gaio,A5079867309
1,Sébastien Herbreteau,A5081838644,Charles Kervrann,A5033807512
2,Sergio Venturini,A5036390189,Raffaella Piccarreta,A5054694299
3,Laura Laguna Salvadó,A5074176483,Bartel A. Van de Walle,A5052763355
4,Zahra Bouanani-Oukhaled,A5047890923,Laurent Breda,A5057081224


In [14]:
df["all_author_openalex_ids"] = df["author_lst"].apply(all_author_openalex_ids)

print("Any-author OpenAlex ID coverage:", (df["all_author_openalex_ids"].apply(len) > 0).mean())
print("Mean available OpenAlex author IDs per paper:", df["all_author_openalex_ids"].apply(len).mean())

df[[
    "first_author_openalex_id",
    "last_author_openalex_id",
    "all_author_openalex_ids"
]].head()


Any-author OpenAlex ID coverage: 1.0
Mean available OpenAlex author IDs per paper: 3.1786438924091036


,first_author_openalex_id,last_author_openalex_id,all_author_openalex_ids
0,A5046716771,A5079867309,"(A5046716771, A5014057822, A5079867309)"
1,A5081838644,A5033807512,"(A5081838644, A5033807512)"
2,A5036390189,A5054694299,"(A5036390189, A5054694299)"
3,A5074176483,A5052763355,"(A5074176483, A5001433532, A5061912935, A50527..."
4,A5047890923,A5057081224,"(A5047890923, A5041988740, A5026270456, A50114..."


In [15]:
def safe_list_len(x):
    return len(x) if isinstance(x, list) else np.nan

df["n_authors"] = df["author_lst"].apply(safe_list_len)

df["n_authors"].describe()

count    3.720575e+06
mean     3.178644e+00
std      1.823737e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      1.270000e+02
Name: n_authors, dtype: float64

## 4. Gender Inference

In [16]:
model = nqg.NBGC()

In [17]:
author_names = pd.unique(
    pd.concat(
        [
            df["first_author_name"],
            df["last_author_name"]
        ],
        ignore_index=True
    ).dropna()
)

author_names = list(author_names)

print("Unique first/last author names:", len(author_names))
author_names[:10]

Unique first/last author names: 1702872


['Christian Sallaberry',
 'Sébastien Herbreteau',
 'Sergio Venturini',
 'Laura Laguna Salvadó',
 'Zahra Bouanani-Oukhaled',
 'Silviu Maniu',
 'Sergei Soloviev 0001',
 'Philippe Blache',
 'Charles de Lacombe',
 'Cyril Grouin']

In [18]:
gender_annotations = model.annotate(author_names, as_df=True)

print("Shape:", gender_annotations.shape)
gender_annotations.head()

Shape: (1702872, 5)


,given,used,sources,counts,p(gf)
0,Christian Sallaberry,christian,33,1080349,0.025
1,Sébastien Herbreteau,sebastien,28,330445,0.000
2,Sergio Venturini,sergio,29,307345,0.000
3,Laura Laguna Salvadó,laura,34,1501112,0.992
4,Zahra Bouanani-Oukhaled,zahra,27,27587,0.990


In [19]:
name_col = gender_annotations.columns[0]

gender_annotations["gender_label"] = gender_annotations["p(gf)"].apply(p_to_label)

gender_annotations[[name_col, "p(gf)", "gender_label"]].head()

,given,p(gf),gender_label
0,Christian Sallaberry,0.025,male
1,Sébastien Herbreteau,0.000,male
2,Sergio Venturini,0.000,male
3,Laura Laguna Salvadó,0.992,female
4,Zahra Bouanani-Oukhaled,0.990,female


In [20]:
name_to_pgf = dict(zip(gender_annotations[name_col], gender_annotations["p(gf)"]))
name_to_gender = dict(zip(gender_annotations[name_col], gender_annotations["gender_label"]))

df["first_author_pgf"] = df["first_author_name"].map(name_to_pgf)
df["first_author_gender"] = df["first_author_name"].map(name_to_gender).fillna("unknown")

df["last_author_pgf"] = df["last_author_name"].map(name_to_pgf)
df["last_author_gender"] = df["last_author_name"].map(name_to_gender).fillna("unknown")

df[[
    "first_author_name",
    "first_author_gender",
    "first_author_pgf",
    "last_author_name",
    "last_author_gender",
    "last_author_pgf"
]].head(10)

,first_author_name,first_author_gender,first_author_pgf,last_author_name,last_author_gender,last_author_pgf
0,Christian Sallaberry,male,0.025,Mauro Gaio,male,0.000
1,Sébastien Herbreteau,male,0.000,Charles Kervrann,male,0.001
2,Sergio Venturini,male,0.000,Raffaella Piccarreta,female,0.989
3,Laura Laguna Salvadó,female,0.992,Bartel A. Van de Walle,male,0.009
4,Zahra Bouanani-Oukhaled,female,0.990,Laurent Breda,male,0.002
5,Silviu Maniu,male,0.003,Bogdan Cautis,male,0.001
6,Sergei Soloviev 0001,male,0.000,Sergei Soloviev 0001,male,0.000
7,Philippe Blache,male,0.036,Philippe Blache,male,0.036
8,Charles de Lacombe,male,0.001,Philippe Lamarre,male,0.036
9,Cyril Grouin,male,0.002,Ludovic Quintard,male,0.002


In [21]:
df["first_last_cat"] = [
    pair_cat(first_gender, last_gender)
    for first_gender, last_gender in zip(
        df["first_author_gender"],
        df["last_author_gender"]
    )
]

df["first_last_cat"].value_counts(dropna=False)

MM    1881450
UM     362908
WM     347031
MU     288987
UU     274113
MW     239869
WW     173028
WU      81863
UW      71326
Name: first_last_cat, dtype: int64

## 5. Prepare Reusable Analysis Columns



In [22]:
df["pub_date"] = pd.to_datetime(df["pub_date"], errors="coerce")
df["year"] = df["pub_date"].dt.year
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

df[["pub_date", "year", "decade"]].head()

,pub_date,year,decade
0,2011-03-16,2011,2010
1,2024-01-01,2024,2020
2,2021-01-01,2021,2020
3,2015-05-24,2015,2010
4,2016-01-01,2016,2010


In [23]:
df["n_refs"] = df["ref_lst"].apply(safe_list_len)
df["n_refs"].describe()

count    3.720575e+06
mean     6.157187e+00
std      7.457754e+00
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      9.000000e+00
max      1.155000e+03
Name: n_refs, dtype: float64

In [24]:
topic_example = df["primary_topic"].dropna().iloc[0]

print(type(topic_example))
topic_example

<class 'dict'>


{'id': 'T10757',
 'name': 'Volunteered Geographic Information and Geospatial Crowdsourcing',
 'subfield': 'Geography, Planning and Development',
 'field': 'Social Sciences',
 'domain': 'Social Sciences'}

In [25]:
df["primary_topic"][0]

{'id': 'T10757',
 'name': 'Volunteered Geographic Information and Geospatial Crowdsourcing',
 'subfield': 'Geography, Planning and Development',
 'field': 'Social Sciences',
 'domain': 'Social Sciences'}

In [26]:
def extract_from_dict(x, key):
    if isinstance(x, dict):
        return x.get(key)
    return None


df["primary_topic_id"] = df["primary_topic"].apply(
    lambda x: extract_from_dict(x, "id")
)

df["primary_topic_name"] = df["primary_topic"].apply(
    lambda x: extract_from_dict(x, "name")
)

df["primary_subfield"] = df["primary_topic"].apply(
    lambda x: extract_from_dict(x, "subfield")
)

df["primary_field"] = df["primary_topic"].apply(
    lambda x: extract_from_dict(x, "field")
)

df["primary_domain"] = df["primary_topic"].apply(
    lambda x: extract_from_dict(x, "domain")
)

df[
    [
        "primary_domain",
        "primary_field",
        "primary_subfield",
        "primary_topic_name",
    ]
].head()

,primary_domain,primary_field,primary_subfield,primary_topic_name
0,Social Sciences,Social Sciences,"Geography, Planning and Development",Volunteered Geographic Information and Geospat...
1,Physical Sciences,Computer Science,Computer Vision and Pattern Recognition,Image Denoising Techniques and Algorithms
2,Physical Sciences,Computer Science,Artificial Intelligence,Model-Based Clustering with Mixture Models
3,Social Sciences,"Business, Management and Accounting",Organizational Behavior and Human Resource Man...,Humanitarian Logistics and Disaster Relief Ope...
4,Physical Sciences,Computer Science,Artificial Intelligence,Semantic Web and Ontology Development


In [27]:
df["venue_rank_clean"] = (
    df["venue_rank"]
    .astype("string")
    .str.upper()
    .replace({"NAN": pd.NA, "NONE": pd.NA})
)

df["venue_rank_clean"] = df["venue_rank_clean"].fillna("N/A")

df["venue_rank_clean"].value_counts(dropna=False)

N/A    1865887
Q1      630140
Q2      315129
B       255514
Q3      161900
A*      150830
C       148328
A       145410
Q4       47437
Name: venue_rank_clean, dtype: Int64

## 6. Citation Edge Filtering and Final Analysis Dataset


In [28]:
REMOVE_SELF_CITATIONS = True
REMOVE_ISOLATED_PAPERS_AFTER_FILTERING = True
REMOVE_COAUTHOR_NEIGHBORHOOD_CITATIONS = True
COAUTHOR_NEIGHBORHOOD_STRICTLY_PRIOR = True
APPLY_RANKED_ONLY_FILTER = False

print("REMOVE_SELF_CITATIONS:", REMOVE_SELF_CITATIONS)
print("REMOVE_ISOLATED_PAPERS_AFTER_FILTERING:", REMOVE_ISOLATED_PAPERS_AFTER_FILTERING)
print("REMOVE_COAUTHOR_NEIGHBORHOOD_CITATIONS:", REMOVE_COAUTHOR_NEIGHBORHOOD_CITATIONS)
print("COAUTHOR_NEIGHBORHOOD_STRICTLY_PRIOR:", COAUTHOR_NEIGHBORHOOD_STRICTLY_PRIOR)
print("APPLY_RANKED_ONLY_FILTER:", APPLY_RANKED_ONLY_FILTER)


REMOVE_SELF_CITATIONS: True
REMOVE_ISOLATED_PAPERS_AFTER_FILTERING: True
REMOVE_COAUTHOR_NEIGHBORHOOD_CITATIONS: True
COAUTHOR_NEIGHBORHOOD_STRICTLY_PRIOR: True
APPLY_RANKED_ONLY_FILTER: False


In [29]:
PAPER_ID_CANDIDATES = [
    "paper_id",
    "id",
    "openalex_id",
    "work_id",
    "paperId",
]

PAPER_ID_COL = next((col for col in PAPER_ID_CANDIDATES if col in df.columns), None)

if PAPER_ID_COL is None:
    raise KeyError(
        "Could not detect a paper ID column. "
        f"Tried: {PAPER_ID_CANDIDATES}. Available columns: {df.columns.tolist()}"
    )

print("Using paper ID column:", PAPER_ID_COL)

df = df[df[PAPER_ID_COL].notna()].copy()
df[PAPER_ID_COL] = df[PAPER_ID_COL].astype(str)

n_before_dedup = len(df)
df = df.drop_duplicates(subset=[PAPER_ID_COL], keep="first").copy()
n_after_dedup = len(df)

print("Papers before ID deduplication:", n_before_dedup)
print("Papers after ID deduplication:", n_after_dedup)
print("Removed duplicate paper IDs:", n_before_dedup - n_after_dedup)

Using paper ID column: id
Papers before ID deduplication: 3720575
Papers after ID deduplication: 3720575
Removed duplicate paper IDs: 0


In [30]:
if APPLY_RANKED_ONLY_FILTER:
    n_before_ranked = len(df)
    df = df[df["venue_rank_clean"].ne("N/A")].copy()
    n_after_ranked = len(df)

    print("Papers before ranked-only filter:", n_before_ranked)
    print("Papers after ranked-only filter:", n_after_ranked)
    print("Removed unranked/N/A venue papers:", n_before_ranked - n_after_ranked)
else:
    print("Ranked-only filter not applied.")

Ranked-only filter not applied.


In [31]:
required_for_edges = [PAPER_ID_COL, "ref_lst"]
missing_for_edges = [col for col in required_for_edges if col not in df.columns]

if missing_for_edges:
    raise KeyError(f"Missing columns required for citation-edge construction: {missing_for_edges}")

paper_ids = set(df[PAPER_ID_COL])


edge_source = df[[PAPER_ID_COL, "ref_lst"]].copy()
edge_source = edge_source[edge_source["ref_lst"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

citation_edges = edge_source.explode("ref_lst", ignore_index=True)
citation_edges = citation_edges.rename(
    columns={
        PAPER_ID_COL: "citing_paper_id",
        "ref_lst": "cited_paper_id",
    }
)

citation_edges = citation_edges[citation_edges["cited_paper_id"].notna()].copy()
citation_edges["citing_paper_id"] = citation_edges["citing_paper_id"].astype(str)
citation_edges["cited_paper_id"] = citation_edges["cited_paper_id"].astype(str)

n_edges_before_internal_filter = len(citation_edges)
citation_edges = citation_edges[citation_edges["cited_paper_id"].isin(paper_ids)].copy()

n_edges_before_dedup = len(citation_edges)
citation_edges = citation_edges.drop_duplicates(
    subset=["citing_paper_id", "cited_paper_id"],
    keep="first"
).copy()

print("Candidate reference edges before internal-paper filter:", n_edges_before_internal_filter)
print("Internal citation edges before deduplication:", n_edges_before_dedup)
print("Internal citation edges after deduplication:", len(citation_edges))
print("Removed duplicate internal edges:", n_edges_before_dedup - len(citation_edges))

citation_edges.head()


Candidate reference edges before internal-paper filter: 22908275
Internal citation edges before deduplication: 22908275
Internal citation edges after deduplication: 22908275
Removed duplicate internal edges: 0


,citing_paper_id,cited_paper_id
0,W2572734975,W238389808
1,W2572734975,W1986370907
2,W2572734975,W1997685381
3,W2572734975,W2060070095
4,W2572734975,W2791342857


In [32]:
filter_counts = {
    "papers_after_basic_preprocessing": len(df),
    "edges_internal_deduplicated": len(citation_edges),
}

if REMOVE_SELF_CITATIONS:
    paper_to_authors = dict(
        zip(
            df[PAPER_ID_COL],
            df["all_author_openalex_ids"].apply(lambda x: frozenset(x) if isinstance(x, (list, tuple, set)) else frozenset())
        )
    )

    citing_ids = citation_edges["citing_paper_id"].to_numpy()
    cited_ids = citation_edges["cited_paper_id"].to_numpy()

    is_self_citation = np.fromiter(
        (
            bool(paper_to_authors.get(citing_id, frozenset()) & paper_to_authors.get(cited_id, frozenset()))
            for citing_id, cited_id in zip(citing_ids, cited_ids)
        ),
        dtype=bool,
        count=len(citation_edges)
    )

    n_self_citation_edges = int(is_self_citation.sum())
    citation_edges = citation_edges.loc[~is_self_citation].copy()

    filter_counts["self_citation_edges_removed"] = n_self_citation_edges
    filter_counts["edges_after_self_citation_removal"] = len(citation_edges)

    print("Removed self-citation edges:", n_self_citation_edges)
    print("Citation edges after self-citation removal:", len(citation_edges))
else:
    filter_counts["self_citation_edges_removed"] = 0
    filter_counts["edges_after_self_citation_removal"] = len(citation_edges)
    print("Self-citation removal not applied.")


Removed self-citation edges: 3041749
Citation edges after self-citation removal: 19866526


### Time-aware coauthor-neighborhood citation filter

This filter removes citation edges that may be driven by close collaboration-network proximity.

For each citation edge `citing_paper -> cited_paper`, the edge is removed only if:

1. one of the **first/last authors** of the citing paper had previously coauthored with
2. at least one author of the cited paper,
3. **before** the citing paper was published.

The filter is therefore time-aware: it does not use coauthorship relationships that only appear after the citing paper's publication year.


In [33]:
if REMOVE_COAUTHOR_NEIGHBORHOOD_CITATIONS:
    required_cols = [
        PAPER_ID_COL,
        "year",
        "all_author_openalex_ids",
        "first_author_openalex_id",
        "last_author_openalex_id",
    ]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing columns required for time-aware coauthor-neighborhood filter: {missing_cols}")

    print("Building time-aware coauthor-neighborhood index...")


    paper_to_year = dict(zip(df[PAPER_ID_COL], pd.to_numeric(df["year"], errors="coerce")))

    def clean_author_list(authors):
        """Return unique, non-null author IDs as strings, preserving no particular order."""
        if not isinstance(authors, (list, tuple, set)):
            return []
        cleaned = []
        seen = set()
        for author_id in authors:
            if pd.isna(author_id):
                continue
            author_id = str(author_id)
            if author_id and author_id not in seen:
                cleaned.append(author_id)
                seen.add(author_id)
        return cleaned

    def clean_author_id(author_id):
        if pd.isna(author_id):
            return None
        author_id = str(author_id)
        return author_id if author_id else None

    paper_to_authors = dict(
        zip(
            df[PAPER_ID_COL],
            df["all_author_openalex_ids"].apply(lambda x: frozenset(clean_author_list(x)))
        )
    )

    paper_to_first_last_authors = {}
    focal_author_universe = set()

    for paper_id, first_author_id, last_author_id in zip(
        df[PAPER_ID_COL],
        df["first_author_openalex_id"],
        df["last_author_openalex_id"],
    ):
        focal_authors = {
            clean_author_id(author_id)
            for author_id in [first_author_id, last_author_id]
            if clean_author_id(author_id) is not None
        }
        focal_authors = frozenset(focal_authors)
        paper_to_first_last_authors[paper_id] = focal_authors
        focal_author_universe.update(focal_authors)

    print("Unique first/last focal authors:", len(focal_author_universe))

   
    author_to_prior_coauthor_year = {}

    paper_author_lists = df["all_author_openalex_ids"].tolist()
    paper_years = pd.to_numeric(df["year"], errors="coerce").tolist()

    for authors, paper_year in zip(paper_author_lists, paper_years):
        if pd.isna(paper_year):
            continue

        authors = clean_author_list(authors)
        if len(authors) < 2:
            continue

        paper_year = int(paper_year)
        author_set = set(authors)

        focal_authors_in_paper = author_set & focal_author_universe
        if not focal_authors_in_paper:
            continue

        for focal_author in focal_authors_in_paper:
            coauthor_years = author_to_prior_coauthor_year.setdefault(focal_author, {})
            for coauthor_id in author_set:
                if coauthor_id == focal_author:
                    continue
                previous_year = coauthor_years.get(coauthor_id)
                if previous_year is None or paper_year < previous_year:
                    coauthor_years[coauthor_id] = paper_year

    print("Focal authors with at least one coauthor:", len(author_to_prior_coauthor_year))

    def has_prior_coauthor_connection(citing_id, cited_id):
        citing_year = paper_to_year.get(citing_id)
        if pd.isna(citing_year):
            return False
        citing_year = int(citing_year)

        citing_focal_authors = paper_to_first_last_authors.get(citing_id, frozenset())
        cited_authors = paper_to_authors.get(cited_id, frozenset())

        if not citing_focal_authors or not cited_authors:
            return False

        for focal_author in citing_focal_authors:
            coauthor_years = author_to_prior_coauthor_year.get(focal_author)
            if not coauthor_years:
                continue

            for cited_author in cited_authors:
                coauthor_year = coauthor_years.get(cited_author)
                if coauthor_year is None:
                    continue

                if COAUTHOR_NEIGHBORHOOD_STRICTLY_PRIOR:
                    if coauthor_year < citing_year:
                        return True
                else:
                    if coauthor_year <= citing_year:
                        return True

        return False

    citing_ids = citation_edges["citing_paper_id"].to_numpy()
    cited_ids = citation_edges["cited_paper_id"].to_numpy()

    is_coauthor_neighborhood = np.fromiter(
        (
            has_prior_coauthor_connection(citing_id, cited_id)
            for citing_id, cited_id in zip(citing_ids, cited_ids)
        ),
        dtype=bool,
        count=len(citation_edges),
    )

    n_coauthor_neighborhood_edges = int(is_coauthor_neighborhood.sum())
    citation_edges = citation_edges.loc[~is_coauthor_neighborhood].copy()

    filter_counts["coauthor_neighborhood_edges_removed"] = n_coauthor_neighborhood_edges
    filter_counts["edges_after_coauthor_neighborhood_removal"] = len(citation_edges)
    filter_counts["coauthor_neighborhood_strictly_prior"] = COAUTHOR_NEIGHBORHOOD_STRICTLY_PRIOR

    print("Removed time-aware coauthor-neighborhood citation edges:", n_coauthor_neighborhood_edges)
    print("Citation edges after coauthor-neighborhood removal:", len(citation_edges))
else:
    filter_counts["coauthor_neighborhood_edges_removed"] = 0
    filter_counts["edges_after_coauthor_neighborhood_removal"] = len(citation_edges)
    filter_counts["coauthor_neighborhood_strictly_prior"] = None
    print("Coauthor-neighborhood citation removal not applied.")


Building time-aware coauthor-neighborhood index...
Unique first/last focal authors: 1912300
Focal authors with at least one coauthor: 1820342
Removed time-aware coauthor-neighborhood citation edges: 961992
Citation edges after coauthor-neighborhood removal: 18904534


In [34]:
if REMOVE_ISOLATED_PAPERS_AFTER_FILTERING:
    active_paper_ids = set(citation_edges["citing_paper_id"]).union(
        set(citation_edges["cited_paper_id"])
    )

    n_papers_before_active_filter = len(df)
    df_final = df[df[PAPER_ID_COL].isin(active_paper_ids)].copy()
    n_papers_after_active_filter = len(df_final)

    filter_counts["papers_before_isolated_removal"] = n_papers_before_active_filter
    filter_counts["papers_after_isolated_removal"] = n_papers_after_active_filter
    filter_counts["isolated_papers_removed"] = n_papers_before_active_filter - n_papers_after_active_filter

    print("Papers before isolated-paper removal:", n_papers_before_active_filter)
    print("Papers after isolated-paper removal:", n_papers_after_active_filter)
    print("Removed isolated papers:", n_papers_before_active_filter - n_papers_after_active_filter)
else:
    df_final = df.copy()
    filter_counts["papers_before_isolated_removal"] = len(df)
    filter_counts["papers_after_isolated_removal"] = len(df_final)
    filter_counts["isolated_papers_removed"] = 0
    print("Isolated-paper removal not applied.")

df = df_final.copy()

final_paper_ids = set(df[PAPER_ID_COL])
citation_edges = citation_edges[
    citation_edges["citing_paper_id"].isin(final_paper_ids)
    & citation_edges["cited_paper_id"].isin(final_paper_ids)
].copy()

filter_counts["edges_final"] = len(citation_edges)
filter_counts["papers_final"] = len(df)

print("Final papers:", len(df))
print("Final citation edges:", len(citation_edges))


Papers before isolated-paper removal: 3720575
Papers after isolated-paper removal: 3026412
Removed isolated papers: 694163
Final papers: 3026412
Final citation edges: 18904534


In [35]:
cs_papers_final = df[df["primary_field"] == "Computer Science"].copy()
cs_paper_ids_final = set(cs_papers_final[PAPER_ID_COL])

citation_edges_all_to_cs_final = citation_edges[
    citation_edges["cited_paper_id"].isin(cs_paper_ids_final)
].copy()

citation_edges_cs_to_cs_final = citation_edges[
    citation_edges["citing_paper_id"].isin(cs_paper_ids_final)
    & citation_edges["cited_paper_id"].isin(cs_paper_ids_final)
].copy()

filter_counts["cs_papers_final"] = len(cs_papers_final)
filter_counts["all_to_cs_edges_final"] = len(citation_edges_all_to_cs_final)
filter_counts["cs_to_cs_edges_final"] = len(citation_edges_cs_to_cs_final)

print("Final CS papers:", len(cs_papers_final))
print("Final All→CS citation edges:", len(citation_edges_all_to_cs_final))
print("Final CS→CS citation edges:", len(citation_edges_cs_to_cs_final))


Final CS papers: 1542691
Final All→CS citation edges: 11601416
Final CS→CS citation edges: 9413932


In [36]:
filter_summary = pd.Series(filter_counts)
filter_summary


papers_after_basic_preprocessing              3720575
edges_internal_deduplicated                  22908275
self_citation_edges_removed                   3041749
edges_after_self_citation_removal            19866526
coauthor_neighborhood_edges_removed            961992
edges_after_coauthor_neighborhood_removal    18904534
coauthor_neighborhood_strictly_prior             True
papers_before_isolated_removal                3720575
papers_after_isolated_removal                 3026412
isolated_papers_removed                        694163
edges_final                                  18904534
papers_final                                  3026412
cs_papers_final                               1542691
all_to_cs_edges_final                        11601416
cs_to_cs_edges_final                          9413932
dtype: object

## 7. Minimal Validation

In [37]:
validation_summary = pd.Series({
    "n_final_papers": len(df),
    "n_final_citation_edges": len(citation_edges),
    "n_final_cs_papers": len(cs_papers_final),
    "n_final_all_to_cs_edges": len(citation_edges_all_to_cs_final),
    "n_final_cs_to_cs_edges": len(citation_edges_cs_to_cs_final),
    "pub_date_coverage_pct": round(df["pub_date"].notna().mean() * 100, 2),
    "year_coverage_pct": round(df["year"].notna().mean() * 100, 2),
    "reference_list_coverage_pct": round(df["ref_lst"].notna().mean() * 100, 2),
    "primary_topic_coverage_pct": round(df["primary_topic"].notna().mean() * 100, 2),
    "primary_field_coverage_pct": round(df["primary_field"].notna().mean() * 100, 2),
    "primary_subfield_coverage_pct": round(df["primary_subfield"].notna().mean() * 100, 2),
    "first_author_name_coverage_pct": round(df["first_author_name"].notna().mean() * 100, 2),
    "last_author_name_coverage_pct": round(df["last_author_name"].notna().mean() * 100, 2),
    "first_author_gender_known_pct": round((df["first_author_gender"] != "unknown").mean() * 100, 2),
    "last_author_gender_known_pct": round((df["last_author_gender"] != "unknown").mean() * 100, 2),
    "known_first_last_category_pct": round(df["first_last_cat"].isin(["MM", "MW", "WM", "WW"]).mean() * 100, 2),
})

validation_summary


n_final_papers                     3026412.00
n_final_citation_edges            18904534.00
n_final_cs_papers                  1542691.00
n_final_all_to_cs_edges           11601416.00
n_final_cs_to_cs_edges             9413932.00
pub_date_coverage_pct                  100.00
year_coverage_pct                      100.00
reference_list_coverage_pct            100.00
primary_topic_coverage_pct             100.00
primary_field_coverage_pct             100.00
primary_subfield_coverage_pct          100.00
first_author_name_coverage_pct         100.00
last_author_name_coverage_pct          100.00
first_author_gender_known_pct           80.26
last_author_gender_known_pct            82.14
known_first_last_category_pct           69.72
dtype: float64

## 8. Save Processed Dataset

The processed dataset is saved after adding first/last author names, first/last OpenAlex author IDs, inferred gender labels, first-last gender category, and number of authors.

In [38]:
papers_final_path = PROCESSED_DIR / "papers_final.pkl"
papers_with_gender_path = PROCESSED_DIR / "papers_with_gender.pkl"
citation_edges_final_path = PROCESSED_DIR / "citation_edges_final.pkl"
citation_edges_all_to_cs_path = PROCESSED_DIR / "citation_edges_all_to_cs_final.pkl"
citation_edges_cs_to_cs_path = PROCESSED_DIR / "citation_edges_cs_to_cs_final.pkl"
filter_summary_path = PROCESSED_DIR / "preprocessing_filter_summary.csv"
validation_summary_path = PROCESSED_DIR / "preprocessing_validation_summary.csv"

df.to_pickle(papers_final_path)

df.to_pickle(papers_with_gender_path)

citation_edges.to_pickle(citation_edges_final_path)
citation_edges_all_to_cs_final.to_pickle(citation_edges_all_to_cs_path)
citation_edges_cs_to_cs_final.to_pickle(citation_edges_cs_to_cs_path)

filter_summary.to_csv(filter_summary_path, header=["value"])
validation_summary.to_csv(validation_summary_path, header=["value"])

print("Saved:", papers_final_path, df.shape)
print("Saved:", papers_with_gender_path, df.shape)
print("Saved:", citation_edges_final_path, citation_edges.shape)
print("Saved:", citation_edges_all_to_cs_path, citation_edges_all_to_cs_final.shape)
print("Saved:", citation_edges_cs_to_cs_path, citation_edges_cs_to_cs_final.shape)
print("Saved:", filter_summary_path)
print("Saved:", validation_summary_path)


Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\papers_final.pkl (3026412, 29)
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\papers_with_gender.pkl (3026412, 29)
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\citation_edges_final.pkl (18904534, 2)
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\citation_edges_all_to_cs_final.pkl (11601416, 2)
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\citation_edges_cs_to_cs_final.pkl (9413932, 2)
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\preprocessing_filter_summary.csv
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\preprocessing_validation_summary.csv


## 8. Save Computer Science Subset

This saves a separate processed dataset containing only papers whose primary field is `Computer Science`.

This subset will be used for CS-focused EDA and later citation-network analyses.

In [39]:
cs_papers = cs_papers_final.copy()

cs_summary = pd.Series({
    "n_final_all_papers": len(df),
    "n_final_cs_papers": len(cs_papers),
    "cs_share_pct": round(len(cs_papers) / len(df) * 100, 2),
    "cs_known_first_last_category_pct": round(
        cs_papers["first_last_cat"].isin(["MM", "MW", "WM", "WW"]).mean() * 100,
        2
    ),
})

cs_summary


n_final_all_papers                  3026412.00
n_final_cs_papers                   1542691.00
cs_share_pct                             50.97
cs_known_first_last_category_pct         70.10
dtype: float64

In [40]:
cs_out_path = PROCESSED_DIR / "cs_papers_prepared.pkl"
cs_final_out_path = PROCESSED_DIR / "cs_papers_final.pkl"

cs_papers.to_pickle(cs_out_path)
cs_papers.to_pickle(cs_final_out_path)

print("Saved:", cs_out_path)
print("Saved:", cs_final_out_path)
print("Shape:", cs_papers.shape)


Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\cs_papers_prepared.pkl
Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\data\processed\cs_papers_final.pkl
Shape: (1542691, 29)


In [41]:
import gc

variables_to_delete = [
    name for name in list(globals())
    if not name.startswith("_")
    and name not in ["gc"]
]

for name in variables_to_delete:
    del globals()[name]

gc.collect()

0